Test 1 — forced_shape_changes
Force shape changes at specific timesteps with no random ones, confirm shape changes happen only there.

In [1]:
from bouncing_ball_task.bouncing_ball import BouncingBallTask

In [2]:
task = BouncingBallTask(
    batch_size=1, sequence_length=20, seed=42,
    forced_shape_changes=[5, 10],
    probability_shape_change=0.0,   # no random shape changes
    return_change=True, return_change_mode='source',
    debug=True,
)
_ = list(task)

shape_change_timesteps = [
    p['time'] for p in task.all_parameters
    if p['shape_change'].any() and p['time'] > 0  # exclude initial changepoint
]
assert shape_change_timesteps == [5, 10], shape_change_timesteps

2026-06-15 15:33:24.893 | DEBUG    | bouncing_ball_task.bouncing_ball:color_mask_mode:594 - Running color_mask_mode setter
2026-06-15 15:33:24.896 | DEBUG    | bouncing_ball_task.bouncing_ball:transitioning_change_mode:572 - Running transitioning_change_mode setter
2026-06-15 15:33:24.897 | DEBUG    | bouncing_ball_task.bouncing_ball:sequence_mode:519 - Running sequence_mode setter
2026-06-15 15:33:24.898 | DEBUG    | bouncing_ball_task.bouncing_ball:resample_change_probabilities:628 - Running resample_change_probabilities
2026-06-15 15:33:24.946 | DEBUG    | bouncing_ball_task.bouncing_ball:color_sampling:476 - Running color_sampling setter
2026-06-15 15:33:24.955 | DEBUG    | bouncing_ball_task.bouncing_ball:return_change_mode:558 - Running return_change_mode setter


DEBUG t=5: color_change_array[5,:,1]=[False], shape_triggered[5]=[False]
DEBUG cooldown mask: [False]
DEBUG after cooldown write: rand_for_color[6,:,1]=[0.9093204]
DEBUG t=6: rand_for_color[6,:,1]=[0.9093204]
DEBUG t=6: pccnvc_fires=[False]


Test 2 — pccosc
Set pccosc=1.0, suppress all other color change sources, force frequent shape changes with no velocity changes. Every shape change should trigger a color change.

In [3]:
task = BouncingBallTask(
    batch_size=4, sequence_length=100, seed=42,
    probability_shape_change=0.5,
    probability_velocity_change=0.0,
    probability_color_change_on_shape_change=1.0,       # pccosc — test target
    probability_color_change_on_velocity_change=0.0,
    probability_color_change_no_velocity_change=0.0,
    probability_color_change_on_velocity_and_shape_change=1.0,
    mask_fraction=None,   # disable grayzone so transitioning_mask never fires
    return_change=True, return_change_mode='source',
    debug=True,
)
_ = list(task)

for p in task.all_parameters:
    if p['time'] == 0:
        continue  # skip initial changepoint
    shape_changed = p['shape_change'][:, 0].astype(bool)
    color_changed = p['color_change'].any(axis=-1)
    assert (color_changed[shape_changed] == True).all()
    assert (color_changed[~shape_changed] == False).all()


2026-06-15 15:33:29.537 | DEBUG    | bouncing_ball_task.bouncing_ball:color_mask_mode:594 - Running color_mask_mode setter
2026-06-15 15:33:29.539 | DEBUG    | bouncing_ball_task.bouncing_ball:transitioning_change_mode:572 - Running transitioning_change_mode setter
2026-06-15 15:33:29.541 | DEBUG    | bouncing_ball_task.bouncing_ball:sequence_mode:519 - Running sequence_mode setter
2026-06-15 15:33:29.542 | DEBUG    | bouncing_ball_task.bouncing_ball:resample_change_probabilities:628 - Running resample_change_probabilities
2026-06-15 15:33:29.544 | DEBUG    | bouncing_ball_task.bouncing_ball:color_sampling:476 - Running color_sampling setter
2026-06-15 15:33:29.551 | DEBUG    | bouncing_ball_task.bouncing_ball:return_change_mode:558 - Running return_change_mode setter


DEBUG t=5: color_change_array[5,:,1]=[False False False  True], shape_triggered[5]=[False False False  True]
DEBUG cooldown mask: [False False False  True]
DEBUG after cooldown write: rand_for_color[6,:,1]=[1. 1. 1. 1.]
DEBUG t=6: rand_for_color[6,:,1]=[1. 1. 1. 1.]
DEBUG t=6: pccnvc_fires=[False False False False]


Test 3 — pccovasc overrides pccovc
Force both a velocity resample and a shape change at the same timestep. Set pccovasc=0, pccovc=1. The color should NOT change despite pccovc=1, because pccovasc takes over when both change.

In [4]:
task = BouncingBallTask(
    batch_size=1, sequence_length=30, seed=42,
    forced_velocity_resamples=[10],
    forced_shape_changes=[10],
    probability_color_change_on_velocity_change=1.0,             # pccovc=1
    probability_color_change_on_velocity_and_shape_change=0.0,   # pccovasc=0 — test target
    probability_color_change_on_shape_change=0.0,
    probability_color_change_no_velocity_change=0.0,
    return_change=True, return_change_mode='source',
    debug=True,
)
_ = list(task)

p = task.all_parameters[10]
assert p['velocity change'].any()     # vel did change
assert p['shape_change'].any()        # shape did change
assert not p['color_change'].any()    # color must NOT change (pccovasc=0)


2026-06-15 15:33:33.732 | DEBUG    | bouncing_ball_task.bouncing_ball:color_mask_mode:594 - Running color_mask_mode setter
2026-06-15 15:33:33.734 | DEBUG    | bouncing_ball_task.bouncing_ball:transitioning_change_mode:572 - Running transitioning_change_mode setter
2026-06-15 15:33:33.735 | DEBUG    | bouncing_ball_task.bouncing_ball:sequence_mode:519 - Running sequence_mode setter
2026-06-15 15:33:33.736 | DEBUG    | bouncing_ball_task.bouncing_ball:resample_change_probabilities:628 - Running resample_change_probabilities
2026-06-15 15:33:33.737 | DEBUG    | bouncing_ball_task.bouncing_ball:color_sampling:476 - Running color_sampling setter
2026-06-15 15:33:33.743 | DEBUG    | bouncing_ball_task.bouncing_ball:return_change_mode:558 - Running return_change_mode setter


DEBUG t=5: color_change_array[5,:,1]=[False], shape_triggered[5]=[False]
DEBUG cooldown mask: [False]
DEBUG after cooldown write: rand_for_color[6,:,1]=[0.04522729]
DEBUG t=6: rand_for_color[6,:,1]=[0.04522729]
DEBUG t=6: pccnvc_fires=[False]


Test 4 — min_t_color_change_after_shape_change
Force a shape change at t=5 with pccosc=1.0 (color always changes). Set min_t_color_change_after_shape_change=10 and min_t_color_change_after_random=1 (very short, so pccnvc can fire freely). Verify no col-1 color changes in t=6–15.

In [6]:
task = BouncingBallTask(
    batch_size=1, sequence_length=50, seed=42,
    forced_shape_changes=[5],
    probability_shape_change=0.0,
    probability_velocity_change=0.0,
    probability_color_change_on_shape_change=1.0,
    probability_color_change_no_velocity_change=0.999,  # pccnvc=1 so cooldown is the only blocker
    probability_color_change_on_velocity_change=0.0,
    probability_color_change_on_velocity_and_shape_change=0.0,
    min_t_color_change_after_shape_change=10,  # test target
    min_t_color_change_after_random=1,
    mask_fraction=None,                  # add this
    return_change=True, return_change_mode='source',
    debug=True,
)
_ = list(task)

for p in task.all_parameters:
    t = p['time']
    if 6 <= t <= 15:
        assert not p['color_change'].any(), f"Unexpected color change at t={t}"


2026-06-15 15:43:11.696 | DEBUG    | bouncing_ball_task.bouncing_ball:color_mask_mode:594 - Running color_mask_mode setter
2026-06-15 15:43:11.722 | DEBUG    | bouncing_ball_task.bouncing_ball:transitioning_change_mode:572 - Running transitioning_change_mode setter
2026-06-15 15:43:11.725 | DEBUG    | bouncing_ball_task.bouncing_ball:sequence_mode:519 - Running sequence_mode setter
2026-06-15 15:43:11.726 | DEBUG    | bouncing_ball_task.bouncing_ball:resample_change_probabilities:628 - Running resample_change_probabilities
2026-06-15 15:43:11.748 | DEBUG    | bouncing_ball_task.bouncing_ball:color_sampling:476 - Running color_sampling setter
2026-06-15 15:43:11.796 | DEBUG    | bouncing_ball_task.bouncing_ball:return_change_mode:558 - Running return_change_mode setter


DEBUG t=5: color_change_array[5,:,1]=[False], shape_triggered[5]=[False]
DEBUG cooldown mask: [False]
DEBUG after cooldown write: rand_for_color[6,:,1]=[0.88721274]
DEBUG t=6: rand_for_color[6,:,1]=[0.88721274]
DEBUG t=6: pccnvc_fires=[False]


The key pattern in all four: zero out every probability you're not testing so failures have an unambiguous cause. task.all_parameters (available with debug=True) gives you the full per-timestep breakdown to inspect.